In [24]:
import pandas as pd
import numpy as np
from unidecode import unidecode
from pathlib import Path
import re

In [25]:
# Buscar a pasta onde está o parquet
pasta = Path("../data/01_bronze/Notificacoes")
arquivo_parquet = next(pasta.glob("*.parquet"))
df = pd.read_parquet(arquivo_parquet)

In [26]:
# Normalizar as linhas
def normalize_rows(series):
    # Substituir nulos por "NaN"
    series = series.fillna(np.nan).astype(str)
    # Remover espaços antes/depois
    series = series.str.strip()
    # Remover acentos/caracteres especiais
    series = series.apply(unidecode)
    # Converter para maiúsculas
    series = series.str.upper()
    return series

for col in df.select_dtypes(include=['object']).columns:
    df[col] = normalize_rows(df[col])

In [27]:
# Converter datas no formato YYYY-MM-DD
def normalize_date_column(series):
    series = series.astype(str).str.strip()
    mask = series.str.match(r"^\d{8}$")  # identifica strings de 8 dígitos
    series.loc[mask] = pd.to_datetime(series[mask], format="%Y%m%d", errors="coerce").dt.strftime("%Y-%m-%d")
    return series

colunas_data = [
    "DATA_INCLUSAO_SISTEMA", "DATA_ULTIMA_ATUALIZACAO", "DATA_NOTIFICACAO", "DATA_NASCIMENTO",
    "DATA_INICIO_HORA", "DATA_FINAL_HORA"
]

for col in colunas_data:
    if col in df.columns:
        df[col] = normalize_date_column(df[col])

In [28]:
# Mapeamento UF numérico
uf_map = {
    "AC": 1, "AL": 2, "AP": 3, "AM": 4, "BA": 5, "CE": 6, "DF": 7, "ES": 8, "GO": 9, "MA": 10, 
    "MT": 11, "MS": 12, "MG": 13, "PA": 14, "PB": 15, "PR": 16, "PE": 17, "PI": 18, "RJ": 19,
    "RN": 20, "RS": 21, "RO": 22, "RR": 23, "SC": 24, "SP": 25, "SE": 26, "TO": 27,
}

# Mapeamento UF descritivo
uf_nome_map = {
    "AC": "ACRE", "AL": "ALAGOAS", "AP": "AMAPA", "AM": "AMAZONAS", "BA": "BAHIA", "CE": "CEARA",
    "DF": "DISTRITO FEDERAL", "ES": "ESPIRITO SANTO", "GO": "GOIAS", "MA": "MARANHAO", "MT": "MATO GROSSO",
    "MS": "MATO GROSSO DO SUL", "MG": "MINAS GERAIS", "PA": "PARA", "PB": "PARAIBA", "PR": "PARANA",
    "PE": "PERNAMBUCO", "PI": "PIAUI", "RJ": "RIO DE JANEIRO", "RN": "RIO GRANDE DO NORTE", "RS": "RIO GRANDE DO SUL",
    "RO": "RONDONIA", "RR": "RORAIMA", "SC": "SANTA CATARINA", "SP": "SAO PAULO", "SE": "SERGIPE", "TO": "TOCANTINS"
}

if "UF" in df.columns:
    df["UF_VALOR"] = df["UF"]  # já normalizada
    # Substitui siglas pelo código, transforma tudo em número ou NaN
    df["UF"] = df["UF_VALOR"].replace(uf_map)
    df["UF"] = pd.to_numeric(df["UF"], errors="coerce").astype("Int64")  # Int64 permite NA
    # Cria coluna de descrição
    df["UF_DESCRICAO"] = df["UF_VALOR"].map(uf_nome_map).fillna("DESCONHECIDO")


In [29]:
# Mapeamento GRUPO_IDADE numérico
grupo_idade_map = {
    "FETO": 1, "NEONATO": 2, "INFANTIL": 3, "CRIANÇA": 4, "ADOLESCENTE": 5,
    "ADULTO": 6, "IDOSO": 7, "OUTRO": 8, "NÃO INFORMADO": 0
}

# Mapeamento descritivo
grupo_idade_descricao = {
    "FETO": "PRE-NASCIMENTO", "NEONATO": "0 A 27 DIAS", "INFANTIL": "28 DIAS A 1 ANO",
    "CRIANÇA": "2 A 11 ANOS", "ADOLESCENTE": "12 A 18 ANOS", "ADULTO": "19 A 59 ANOS",
    "IDOSO": "60 ANOS OU MAIS", "OUTRO": "DESCONHECIDO", "NÃO INFORMADO": "DESCONHECIDO"
}

if "GRUPO_IDADE" in df.columns:
    # Mantém coluna original
    df["GRUPO_IDADE_VALOR"] = df["GRUPO_IDADE"]
    
    # Converte para código numérico
    df["GRUPO_IDADE"] = df["GRUPO_IDADE_VALOR"].replace(grupo_idade_map)
    df["GRUPO_IDADE"] = pd.to_numeric(df["GRUPO_IDADE"], errors="coerce").fillna(0).astype(int)
    
    # Cria coluna de descrição
    df["GRUPO_IDADE_DESCRICAO"] = df["GRUPO_IDADE_VALOR"].map(grupo_idade_descricao)
    df["GRUPO_IDADE_DESCRICAO"] = df["GRUPO_IDADE_DESCRICAO"].fillna("DESCONHECIDO")

In [30]:
def mapear_coluna(df, coluna, mapa, nome_valor=None):
    if coluna not in df.columns:
        return df
    
    nome_valor = nome_valor or f"{coluna}_VALOR"
    
    # Cria coluna com o texto original
    df[nome_valor] = df[coluna]
    
    # Substitui pelos códigos
    df[coluna] = df[coluna].replace(mapa)
    
    # Converte para inteiro
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce").fillna(0).astype(int)
    
    return df

In [31]:
# Mapeamento TIPO_ENTRADA_VIGIMED
tipo_entrada_vigimed_map = {
    "SERVICOS DE SAUDE": 1, "EMPRESAS FARMACEUTICAS": 2, "PACIENTES E PROFISSIONAIS DE SAUDE": 3,
    "SERVICOS DE VACINACAO": 4, "EREPORTING - INDUSTRIA, ENTRADA MANUAL DE DADOS": 5,
    "EREPORTING - INDUSTRIA, CARGA E2B": 6, "VIGIMOBILE": 7, "VIGIFLOW EFORMS": 8,
}

df = mapear_coluna(df, "TIPO_ENTRADA_VIGIMED", tipo_entrada_vigimed_map)

In [32]:
# Mapeamento RECEBIDO_DE
recebido_de_map = {
    "CARGA E2B": 1, "ENTRADA MANUAL DE DADOS": 2, "AUTORIDADE REGULADORA": 3,
    "CENTRO REGIONAL DE FARMACOVIGILANCIA": 4, "EMPRESA FARMACEUTICA": 5,
    "OUTRO (P.EX. DISTRIBUIDORA, FINANCIADOR DE ESTUDO, ORGANIZACAO REPRESENTATIVA PARA PESQUISA CLINICA, OU ORGANIZACAO NAO-COMERCIAL)": 6,
    "PACIENTE/CONSUMIDOR": 7, "PROFISSIONAL DE SAUDE": 8,
}

df = mapear_coluna(df, "RECEBIDO_DE", recebido_de_map)

In [33]:
# Mapeamento TIPO_NOTIFICACAO
tipo_notificacao_map = {
    "NOTIFICACAO ESPONTANEA": 1, "NOTIFICACAO DE ESTUDO": 2, "OUTRO": 3,
    "NAO DISPONIVEL PELO NOTIFICADOR (DESCONHECIDO)": 4,
}

df = mapear_coluna(df, "TIPO_NOTIFICACAO", tipo_notificacao_map)

C:\Users\janet\AppData\Local\Temp\ipykernel_14380\2764533706.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[coluna] = df[coluna].replace(mapa)


In [34]:
# Mapeamento SEXO
sexo_map = {
    "FEMININO": 1, "MASCULINO": 2, "DESCONHECIDO": 0,
}

df = mapear_coluna(df, "SEXO", sexo_map)

In [35]:
# Mapeamento GRAVIDADE
gravidade_map = {
    "OUTRO EFEITO CLINICAMENTE SIGNIFICATIVO": 1, "HOSPITALIZACAO": 2, "AMEACA R VIDA": 3,
    "RESULTOU EM OBITO": 4, "INCAPACIDADE PERSISTENTE OU SIGNIFICATIVA": 5,
    "ANOMALIA CONGENITA OU MALFORMACAO AO NASCER": 6, "NAO INFORMADO": 0,
}

df = mapear_coluna(df, "GRAVIDADE", gravidade_map)

In [36]:
# Mapeamento DESFECHO
desfecho_map = {
    "RECUPERADO": 1, "DESCONHECIDO": 2, "EM RECUPERACAO": 4, "NAO RECUPERADO": 5, "FATAL": 6,
}

df = mapear_coluna(df, "DESFECHO", desfecho_map)

In [37]:
# Mapeamento RELACAO_MEDICAMENTO_EVENTO
relacao_medicamento_evento_map = {
    "SUSPEITO": 1, "CONCOMITANTE": 2, "INTERACAO": 3, "MEDICAMENTO NAO ADMINISTRADO": 4,
}

df = mapear_coluna(df, "RELACAO_MEDICAMENTO_EVENTO", relacao_medicamento_evento_map)

In [38]:
# Mapeamento ACAO_ADOTADA
acao_adotada_map = {
    "AUMENTO DA DOSE": 1, "DESCONHECIDO": 2, "NAO APLICAVEL": 3, "REDUCAO DA DOSE": 4,
    "RETIRADA DO MEDICAMENTO": 5, "SEM ALTERACAO DA DOSE": 6,
}

df = mapear_coluna(df, "ACAO_ADOTADA", acao_adotada_map)

In [39]:
# Mapeamento NOTIFICADOR
notificador_map = {
    "ADVOGADO": 1, "CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE": 2, "FARMACEUTICO": 3, "MEDICO": 4,
    "OUTRO PROFISSIONAL DE SAUDE": 5,
}

df = mapear_coluna(df, "NOTIFICADOR", notificador_map)

In [40]:
def normalize_sim_nao(df, coluna):
    if coluna in df.columns:
        # Normaliza: maiúscula, remove espaços
        df[coluna] = df[coluna].astype(str).str.strip().str.upper()
        
        # Cria coluna com texto original
        df[coluna + "_VALOR"] = df[coluna]
        
        # Substitui tudo que não for SIM/NAO/NAO INFORMADO por NAO INFORMADO
        df[coluna + "_VALOR"] = df[coluna + "_VALOR"].where(
            df[coluna + "_VALOR"].isin(["SIM", "NAO", "NAO INFORMADO"]), 
            "NAO INFORMADO"
        )
        
        # Mapeamento SIM/NAO/NAO INFORMADO
        mapa = {"SIM": 1, "NAO": 2, "NAO INFORMADO": 0}
        
        # Substitui a coluna principal pelos códigos
        df[coluna] = df[coluna + "_VALOR"].replace(mapa)
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce").fillna(0).astype(int)
    
    return df

# Lista de colunas SIM/NAO/NAO INFORMADO
colunas = ["NOTIFICACAO_PARENT_CHILD", "GESTANTE", "LACTANTE", "GRAVE"]

for col in colunas:
    df = normalize_sim_nao(df, col)


C:\Users\janet\AppData\Local\Temp\ipykernel_14380\3430108259.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[coluna] = df[coluna + "_VALOR"].replace(mapa)
C:\Users\janet\AppData\Local\Temp\ipykernel_14380\3430108259.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[coluna] = df[coluna + "_VALOR"].replace(mapa)
C:\Users\janet\AppData\Local\Temp\ipykernel_14380\3430108259.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly

In [41]:
def extrair_numero(df, coluna, nome_valor=None, nome_unidade=None):
    if coluna not in df.columns:
        return df
    
    nome_valor = nome_valor or f"{coluna}_VALOR"
    nome_unidade = nome_unidade or f"{coluna}_UNIDADE"
    
    def extrair(valor):
        if pd.isna(valor):
            return (np.nan, np.nan)
        texto = str(valor).strip().upper()
        match = re.match(r"(\d+)\s*([A-ZÇÃÉÍ]+)?", texto)
        if not match:
            return (np.nan, np.nan)
        numero = int(match.group(1))
        unidade = match.group(2) if match.group(2) else ""
        unidade = (
            "ANO(S)" if "ANO" in unidade else
            "MES(ES)" if "MES" in unidade else
            "DIA(S)" if "DIA" in unidade else "NAN"
        )
        return (numero, unidade)
    
    df[[nome_valor, nome_unidade]] = df[coluna].apply(lambda x: pd.Series(extrair(x)))
    df[nome_valor] = df[nome_valor].fillna(0).astype("Int64")
    
    return df

df = extrair_numero(df, "IDADE_MOMENTO_REACAO")
df = extrair_numero(df, "DURACAO")

In [42]:
df.head()

,UF,TIPO_ENTRADA_VIGIMED,RECEBIDO_DE,IDENTIFICACAO_NOTIFICACAO,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,TIPO_NOTIFICACAO,NOTIFICACAO_PARENT_CHILD,DATA_NASCIMENTO,...,ACAO_ADOTADA_VALOR,NOTIFICADOR_VALOR,NOTIFICACAO_PARENT_CHILD_VALOR,GESTANTE_VALOR,LACTANTE_VALOR,GRAVE_VALOR,IDADE_MOMENTO_REACAO_VALOR,IDADE_MOMENTO_REACAO_UNIDADE,DURACAO_VALOR,DURACAO_UNIDADE
0,25,2,5,BR-ANVISA-300212656,2023-09-28,2023-09-28,NAN,1,0,1990-01-31,...,NAO APLICAVEL,MEDICO,NAO INFORMADO,NAO,NAO,SIM,30,ANO(S),12,DIA(S)
1,25,2,5,BR-ANVISA-300208322,2023-09-01,2023-09-01,NAN,1,0,NAN,...,NAO APLICAVEL,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,NAO INFORMADO,NAO,NAO,NAO,0,NaN,0,NaN
2,25,2,5,BR-ANVISA-300214015,2023-10-06,2023-10-06,NAN,1,0,1971-05-22,...,NAN,MEDICO,NAO INFORMADO,NAO,NAO,SIM,49,ANO(S),0,NaN
3,25,6,5,BR-ANVISA-300345102,2025-07-21,2025-07-21,NAN,1,0,1974-11-23,...,DESCONHECIDO,FARMACEUTICO,NAO INFORMADO,NAO,NAO,SIM,46,ANO(S),0,NaN
4,25,2,5,BR-ANVISA-300212385,2023-09-27,2023-09-27,NAN,1,0,NAN,...,NAN,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,NAO INFORMADO,NAO,NAO,SIM,0,NaN,0,NaN


In [43]:
idx = df.columns.get_loc("DATA_NASCIMENTO")
df.iloc[:, idx:].head()


,DATA_NASCIMENTO,IDADE_MOMENTO_REACAO,GRUPO_IDADE,IDADE_GESTACIONAL_MOMENTO_REACAO,SEXO,GESTANTE,LACTANTE,PESO_KG,ALTURA_CM,REACAO_EVENTO_ADVERSO_MEDDRA,...,ACAO_ADOTADA_VALOR,NOTIFICADOR_VALOR,NOTIFICACAO_PARENT_CHILD_VALOR,GESTANTE_VALOR,LACTANTE_VALOR,GRAVE_VALOR,IDADE_MOMENTO_REACAO_VALOR,IDADE_MOMENTO_REACAO_UNIDADE,DURACAO_VALOR,DURACAO_UNIDADE
0,1990-01-31,30 ANO,0,NAN,1,2,2,68.0,165,HEMIPARESIA,...,NAO APLICAVEL,MEDICO,NAO INFORMADO,NAO,NAO,SIM,30,ANO(S),12,DIA(S)
1,NAN,NAN,0,NAN,1,2,2,NAN,NAN,CEFALEIA,...,NAO APLICAVEL,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,NAO INFORMADO,NAO,NAO,NAO,0,NaN,0,NaN
2,1971-05-22,49 ANO,0,NAN,2,2,2,NAN,NAN,NEFROLITIASE,...,NAN,MEDICO,NAO INFORMADO,NAO,NAO,SIM,49,ANO(S),0,NaN
3,1974-11-23,46 ANO,0,NAN,2,2,2,NAN,NAN,CHOQUE ANAFILATICO,...,DESCONHECIDO,FARMACEUTICO,NAO INFORMADO,NAO,NAO,SIM,46,ANO(S),0,NaN
4,NAN,NAN,0,NAN,2,2,2,NAN,NAN,ABSCESSO NASAL,...,NAN,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,NAO INFORMADO,NAO,NAO,SIM,0,NaN,0,NaN


In [44]:
idx = df.columns.get_loc("REACAO_EVENTO_ADVERSO_MEDDRA")
df.iloc[:, idx:].head()

,REACAO_EVENTO_ADVERSO_MEDDRA,GRAVE,GRAVIDADE,DESFECHO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,ACAO_ADOTADA,...,ACAO_ADOTADA_VALOR,NOTIFICADOR_VALOR,NOTIFICACAO_PARENT_CHILD_VALOR,GESTANTE_VALOR,LACTANTE_VALOR,GRAVE_VALOR,IDADE_MOMENTO_REACAO_VALOR,IDADE_MOMENTO_REACAO_UNIDADE,DURACAO_VALOR,DURACAO_UNIDADE
0,HEMIPARESIA,1,1,1,2021-01-25,2021-02-06,12 DIA,2,TAMISA,3,...,NAO APLICAVEL,MEDICO,NAO INFORMADO,NAO,NAO,SIM,30,ANO(S),12,DIA(S)
1,CEFALEIA,2,1,2,2021-01-22,NAN,NAN,1,CORONAVAC,3,...,NAO APLICAVEL,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,NAO INFORMADO,NAO,NAO,NAO,0,NaN,0,NaN
2,NEFROLITIASE,1,2,2,2021-02-03,NAN,NAN,2,METFORMINA,0,...,NAN,MEDICO,NAO INFORMADO,NAO,NAO,SIM,49,ANO(S),0,NaN
3,CHOQUE ANAFILATICO,1,3,2,2021-03-02,NAN,NAN,1,SORO ANTIBOTROPICO (PENTAVALENTE),2,...,DESCONHECIDO,FARMACEUTICO,NAO INFORMADO,NAO,NAO,SIM,46,ANO(S),0,NaN
4,ABSCESSO NASAL,1,1,2,NAN,NAN,NAN,1,VACINA ADSORVIDA COVID-19 (INATIVADA),0,...,NAN,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,NAO INFORMADO,NAO,NAO,SIM,0,NaN,0,NaN


In [45]:
idx = df.columns.get_loc("ACAO_ADOTADA")
df.iloc[:, idx:].head()

,ACAO_ADOTADA,NOTIFICADOR,UF_VALOR,UF_DESCRICAO,GRUPO_IDADE_VALOR,GRUPO_IDADE_DESCRICAO,TIPO_ENTRADA_VIGIMED_VALOR,RECEBIDO_DE_VALOR,TIPO_NOTIFICACAO_VALOR,SEXO_VALOR,...,ACAO_ADOTADA_VALOR,NOTIFICADOR_VALOR,NOTIFICACAO_PARENT_CHILD_VALOR,GESTANTE_VALOR,LACTANTE_VALOR,GRAVE_VALOR,IDADE_MOMENTO_REACAO_VALOR,IDADE_MOMENTO_REACAO_UNIDADE,DURACAO_VALOR,DURACAO_UNIDADE
0,3,4,SP,SAO PAULO,NAN,DESCONHECIDO,EMPRESAS FARMACEUTICAS,EMPRESA FARMACEUTICA,NOTIFICACAO ESPONTANEA,FEMININO,...,NAO APLICAVEL,MEDICO,NAO INFORMADO,NAO,NAO,SIM,30,ANO(S),12,DIA(S)
1,3,2,SP,SAO PAULO,NAN,DESCONHECIDO,EMPRESAS FARMACEUTICAS,EMPRESA FARMACEUTICA,NOTIFICACAO ESPONTANEA,FEMININO,...,NAO APLICAVEL,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,NAO INFORMADO,NAO,NAO,NAO,0,NaN,0,NaN
2,0,4,SP,SAO PAULO,NAN,DESCONHECIDO,EMPRESAS FARMACEUTICAS,EMPRESA FARMACEUTICA,NOTIFICACAO ESPONTANEA,MASCULINO,...,NAN,MEDICO,NAO INFORMADO,NAO,NAO,SIM,49,ANO(S),0,NaN
3,2,3,SP,SAO PAULO,NAN,DESCONHECIDO,"EREPORTING - INDUSTRIA, CARGA E2B",EMPRESA FARMACEUTICA,NOTIFICACAO ESPONTANEA,MASCULINO,...,DESCONHECIDO,FARMACEUTICO,NAO INFORMADO,NAO,NAO,SIM,46,ANO(S),0,NaN
4,0,2,SP,SAO PAULO,NAN,DESCONHECIDO,EMPRESAS FARMACEUTICAS,EMPRESA FARMACEUTICA,NOTIFICACAO ESPONTANEA,MASCULINO,...,NAN,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,NAO INFORMADO,NAO,NAO,SIM,0,NaN,0,NaN


In [46]:
idx = df.columns.get_loc("DESFECHO_VALOR")
df.iloc[:, idx:].head()

,DESFECHO_VALOR,RELACAO_MEDICAMENTO_EVENTO_VALOR,ACAO_ADOTADA_VALOR,NOTIFICADOR_VALOR,NOTIFICACAO_PARENT_CHILD_VALOR,GESTANTE_VALOR,LACTANTE_VALOR,GRAVE_VALOR,IDADE_MOMENTO_REACAO_VALOR,IDADE_MOMENTO_REACAO_UNIDADE,DURACAO_VALOR,DURACAO_UNIDADE
0,RECUPERADO,CONCOMITANTE,NAO APLICAVEL,MEDICO,NAO INFORMADO,NAO,NAO,SIM,30,ANO(S),12,DIA(S)
1,DESCONHECIDO,SUSPEITO,NAO APLICAVEL,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,NAO INFORMADO,NAO,NAO,NAO,0,NaN,0,NaN
2,DESCONHECIDO,CONCOMITANTE,NAN,MEDICO,NAO INFORMADO,NAO,NAO,SIM,49,ANO(S),0,NaN
3,DESCONHECIDO,SUSPEITO,DESCONHECIDO,FARMACEUTICO,NAO INFORMADO,NAO,NAO,SIM,46,ANO(S),0,NaN
4,DESCONHECIDO,SUSPEITO,NAN,CONSUMIDOR OU OUTRO NAO PROFISSIONAL DE SAUDE,NAO INFORMADO,NAO,NAO,SIM,0,NaN,0,NaN
